# nano-triton: a from-scratch Triton + Helion, in one notebook

[`nano-triton`](https://pypi.org/project/nano-triton/) is a miniature rebuild of the modern GPU kernel stack:

- **newt** is a nano-Triton: write a kernel as a Python function over a *block* of data, and it compiles to real GPU machine code (Python AST -> CUDA C++ -> NVRTC -> cubin), no MLIR or LLVM.
- **deuteron** is a nano-Helion: write PyTorch-like *tiles*, and it generates a newt kernel and autotunes it.

This notebook installs it and runs three things: a vector add, a fused softmax, and an autotuned tensor-core matmul.

> **You need an NVIDIA GPU.** In Google Colab: **Runtime -> Change runtime type -> GPU**. Timing numbers below depend on the GPU you run on.

Docs and full writeup: https://arpitsinghgautam.me/nano-triton/ &nbsp;|&nbsp; Source: https://github.com/arpitsinghgautam/nano-triton

In [ ]:
!pip install nano-triton

In [1]:
import warnings
warnings.filterwarnings("ignore")

import torch
import newt
import newt.language as nl
import deuteron as dt

assert torch.cuda.is_available(), "No GPU found. In Colab: Runtime -> Change runtime type -> GPU."
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

torch 2.11.0+cu128 | GPU: NVIDIA RTX PRO 5000 Blackwell Generation Laptop GPU


## 1. Your first newt kernel: vector add

A newt kernel is a Triton kernel with `tl` swapped for `nl`. Each program instance (one CUDA thread block) handles a `BLOCK` of elements; the mask keeps the ragged tail safe when the size is not a multiple of `BLOCK`.

In [2]:
@newt.jit
def add_kernel(x_ptr, y_ptr, out_ptr, n, BLOCK: nl.constexpr):
    pid = nl.program_id(0)
    offs = pid * BLOCK + nl.arange(0, BLOCK)
    mask = offs < n
    x = nl.load(x_ptr + offs, mask=mask)
    y = nl.load(y_ptr + offs, mask=mask)
    nl.store(out_ptr + offs, x + y, mask=mask)

n = 1 << 24
x = torch.randn(n, device="cuda")
y = torch.randn(n, device="cuda")
out = torch.empty_like(x)

grid = lambda m: (newt.cdiv(n, m["BLOCK"]),)
add_kernel[grid](x, y, out, n, BLOCK=4096)

print("matches torch:", torch.allclose(out, x + y))

t = newt.testing.do_bench(lambda: add_kernel[grid](x, y, out, n, BLOCK=4096))
print(f"{n:,} elements in {t:.3f} ms ({3 * n * 4 / t / 1e6:.0f} GB/s)")

matches torch: True
16,777,216 elements in 0.315 ms (639 GB/s)


## 2. A fused softmax

One program per row. The whole row is loaded once, the max and sum reductions run through warp shuffles and shared memory, and the row is normalized on-chip: three global-memory passes in eager PyTorch, fused into one here. Note the row length is not a power of two; the mask handles the edge.

In [3]:
@newt.jit
def softmax_kernel(x_ptr, out_ptr, N, stride, BLOCK_N: nl.constexpr):
    row = nl.program_id(0)
    cols = nl.arange(0, BLOCK_N)
    mask = cols < N
    x = nl.load(x_ptr + row * stride + cols, mask=mask, other=float("-inf"))
    x = x - nl.max(x, axis=0)
    num = nl.exp(x)
    den = nl.sum(num, axis=0)
    nl.store(out_ptr + row * stride + cols, num / den, mask=mask)

x = torch.randn(4096, 2500, device="cuda")     # non-power-of-two row length
out = torch.empty_like(x)
BLOCK_N = newt.next_power_of_2(x.shape[1])
M, N, s = x.shape[0], x.shape[1], x.stride(0)

softmax_kernel[(M,)](x, out, N, s, BLOCK_N=BLOCK_N, num_warps=4)

ref = torch.softmax(x, dim=-1)
print("matches torch:", torch.allclose(out, ref, atol=1e-6))

t_newt = newt.testing.do_bench(lambda: softmax_kernel[(M,)](x, out, N, s, BLOCK_N=BLOCK_N, num_warps=4))
t_torch = newt.testing.do_bench(lambda: torch.softmax(x, -1))
print(f"newt {t_newt:.3f} ms | torch {t_torch:.3f} ms")

matches torch: True
newt 0.132 ms | torch 0.142 ms


## 3. deuteron: write tiles, get an autotuned tensor-core kernel

deuteron is the Helion layer. You write tile-level code with no program ids, offsets, masks or block sizes. It traces the function, generates a newt kernel, and autotunes it against an eager-PyTorch oracle, so a configuration that computes the wrong answer is discarded before it is ever timed.

In [4]:
@dt.kernel(verbose=False)
def matmul(x, y, out):
    for tile_m, tile_n in dt.tile([x.shape[0], y.shape[1]]):
        acc = dt.zeros([tile_m, tile_n], dtype=dt.float32)
        for tile_k in dt.tile(x.shape[1]):
            acc += x[tile_m, tile_k] @ y[tile_k, tile_n]
        out[tile_m, tile_n] = acc

M = K = N = 512
x = torch.randn(M, K, device="cuda", dtype=torch.float16)
y = torch.randn(K, N, device="cuda", dtype=torch.float16)
out = torch.empty(M, N, device="cuda", dtype=torch.float16)

matmul(x, y, out)     # first call: trace -> autotune -> cache -> launch

ref = (x.float() @ y.float()).half()
print("matches torch:", torch.allclose(out.float(), ref.float(), atol=1e-1, rtol=1e-2))
print("best config:", dict(matmul.best_config))

matches torch: True
best config: {'BLOCK_TILE_M': 64, 'BLOCK_TILE_N': 64, 'BLOCK_TILE_K': 128, 'num_warps': 8, 'num_stages': 2}


### The kernel deuteron generated

All of that came from those few lines of tile code. Here is the newt kernel it actually generated and tuned, tile sizes baked in as compile-time constants:

In [5]:
print(matmul.to_newt_source(x, y, out))

import newt
import newt.language as nl

@newt.jit
def matmul_newt(x_ptr, x_d0, x_d1, x_s0, x_s1, y_ptr, y_d0, y_d1, y_s0, y_s1, out_ptr, out_d0, out_d1, out_s0, out_s1, BLOCK_TILE_M: nl.constexpr, BLOCK_TILE_N: nl.constexpr, BLOCK_TILE_K: nl.constexpr):
    pid_0 = nl.program_id(0)
    offs_tile_m = pid_0 * BLOCK_TILE_M + nl.arange(0, BLOCK_TILE_M)
    mask_tile_m = offs_tile_m < x_d0
    pid_1 = nl.program_id(1)
    offs_tile_n = pid_1 * BLOCK_TILE_N + nl.arange(0, BLOCK_TILE_N)
    mask_tile_n = offs_tile_n < y_d1
    acc = nl.zeros((BLOCK_TILE_M, BLOCK_TILE_N,), dtype=nl.float32)
    for _i_tile_k in range(0, nl.cdiv(x_d1, BLOCK_TILE_K)):
        offs_tile_k = _i_tile_k * BLOCK_TILE_K + nl.arange(0, BLOCK_TILE_K)
        mask_tile_k = offs_tile_k < x_d1
        acc = nl.dot(nl.load(x_ptr + offs_tile_m[:, None] * x_s0 + offs_tile_k[None, :] * x_s1, mask=mask_tile_m[:, None] & mask_tile_k[None, :], other=0.0), nl.load(y_ptr + offs_tile_k[:, None] * y_s0 + offs_tile_n[None, :] * y_s1, 

## Where to go next

- **Docs and the full writeup:** https://arpitsinghgautam.me/nano-triton/
- **Source and more examples:** https://github.com/arpitsinghgautam/nano-triton
- **Install:** `pip install nano-triton`

The repo has the rest of the example progression (layernorm, fused flash attention) and a from-zero explainer of how the compiler and autotuner work.